# 1. Data: which films we keep, and why

**Question.** Which films can we measure a return for?

**Measure.** Return is ln(gross / budget). 0 means the gross equalled the budget.

**Method.** Start from the IMDb films with 10,000+ votes, keep those with comparable money, then those that also have a usable screenplay. The money rule is in one module, `src/analysis/clean_rule.py`, and the columns are described in `data/ANALYSIS_SET.md`.

**What it misses.** Films under 10,000 votes, films without both figures, and everything a budget leaves out (marketing) or a gross leaves out (home video, streaming). Gross over budget isn't profit.

In [1]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "analysis").exists())
A = ROOT / "src" / "analysis"
pd.set_option("display.max_colwidth", 90)

## The film sets

In [2]:
links = pd.read_csv(ROOT / "data" / "movie_links.csv")
cov = pd.read_csv(A / "screenplays/word2vec/results/prerelease_coverage.csv")
h2h = json.loads((A / "compare/outputs/head_to_head/summary.json").read_text())
pd.Series({"IMDb films with 10,000+ votes": links.imdb_id.nunique(),
           "with comparable budget and gross": int(cov[cov.group == "all films"].films.sum()),
           "also with a usable screenplay (common set)": h2h["films"]}).to_frame("films")

,films
"IMDb films with 10,000+ votes",12673
with comparable budget and gross,6346
also with a usable screenplay (common set),1598


## Who has a screenplay?

Scripts exist mainly for popular films. The table compares films with money that do and don't have one (any MovieSum script, before the token rule).

In [3]:
cov[cov.group == "all films"][["label", "films", "median_votes", "median_budget_musd", "mean_ln_return"]]

,label,films,median_votes,median_budget_musd,mean_ln_return
0,script,1621,159586,25.0,1.008760
1,no script,4725,45386,20.0,0.423369


The share of hits and flops that have a script, with a film counted once per genre. The gap is compared with 2,000 shuffles of the labels within genre and era.

In [4]:
d = cov[cov.group == "all film-genre pairs"].pivot(index="tier", columns="label", values="coverage")[["hit", "middle", "flop"]]
gap = pd.read_csv(A / "screenplays/word2vec/results/prerelease_coverage_gap.csv").query("group == 'all film-genre pairs'").set_index("tier")
d.join(gap[["gap", "null_lo", "null_hi", "p_two_sided"]]).round(3)

,hit,middle,flop,gap,null_lo,null_hi,p_two_sided
tier,,,,,,,
100k+,0.494,0.420,0.421,0.072,-0.025,0.075,0.074
10k-100k,0.142,0.162,0.107,0.035,-0.014,0.022,0.001
all,0.365,0.262,0.135,0.230,-0.018,0.018,0.000


## A second source for the money

The Wikipedia figures compared with TMDB's, from `src/scripts/imdb/tmdb_compare.py`.

In [5]:
t = json.loads((A / "checks/tmdb_compare.json").read_text())
pd.Series({k: v for k, v in t.items() if not isinstance(v, (dict, list))}).to_frame("value")

,value
tmdb_source,tmdb/tmdb_project_movies.csv
tmdb_snapshot,2026-11-15
clean_movies,6414
matched_to_tmdb_row,6414
tmdb_revenue_under_1000,4
tmdb_budget_under_1000,0
identical_tolerance,0.005
near_break_even_factor,1.25
all_movies,12673


## Were dialogue and description split correctly?

One of us read a sample of script elements by hand and checked the layer each was put in.

In [6]:
pd.read_csv(A / "screenplays/data_check/manual_check_summary.csv")

,sheet,group,layer,k,n,share,ci_low,ci_high,what
0,main,1990,description,1,60,0.017,0.003,0.089,points judged wrong by the parser's layer
1,main,1990,dialogue,0,60,0.000,0.000,0.060,points judged wrong by the parser's layer
2,main,1990,other,0,28,0.000,0.000,0.121,points judged wrong by the parser's layer
3,main,1990,parenthetical,0,2,0.000,0.000,0.658,points judged wrong by the parser's layer
4,main,pre-1990,description,0,40,0.000,0.000,0.088,points judged wrong by the parser's layer
5,main,pre-1990,dialogue,2,40,0.050,0.014,0.165,points judged wrong by the parser's layer
6,main,pre-1990,other,0,16,0.000,0.000,0.194,points judged wrong by the parser's layer
7,main,pre-1990,parenthetical,0,4,0.000,0.000,0.490,points judged wrong by the parser's layer
8,redflags,all,caps_run_in_dialogue,0,10,0.000,0.000,0.278,"real problems among suspicious elements (hunting list, NOT an error rate); other answe..."
9,redflags,all,heading_in_dialogue,7,7,1.000,0.646,1.000,"real problems among suspicious elements (hunting list, NOT an error rate); other answe..."
